# JYXFE feature cleaning: DIM reduction
gold

In [1]:
import pandas as pd

df_feat_ictalVspreictal_1min = pd.read_pickle("/home/tperezsanchez/FoundationModel_EEG_Dissertation/Main_project/results/JYXFE/Feature_ext/Part2_features/dfJYXFE_FEAT_ICTvsPRE_1min_SELECTEDMANN.pkl")

print(df_feat_ictalVspreictal_1min.head())


                   file_name  window_id  start_sample  end_sample          fs  \
0  JYXFE_22_preproc_full.npz        983       2034810     2036880  207.031055   
1  JYXFE_22_preproc_full.npz        984       2036880     2038950  207.031055   
2  JYXFE_22_preproc_full.npz        985       2038950     2041020  207.031055   
3  JYXFE_22_preproc_full.npz        986       2041020     2043090  207.031055   
4  JYXFE_22_preproc_full.npz        987       2043090     2045160  207.031055   

   n_channels                seizure_onsets             window_start_time  \
0           2  [2021-05-04 20:33:31.000000] 2021-05-04 20:27:26.525499679   
1           2  [2021-05-04 20:33:31.000000] 2021-05-04 20:27:36.523999679   
2           2  [2021-05-04 20:33:31.000000] 2021-05-04 20:27:46.522499678   
3           2  [2021-05-04 20:33:31.000000] 2021-05-04 20:27:56.520999678   
4           2  [2021-05-04 20:33:31.000000] 2021-05-04 20:28:06.519499678   

                window_end_time  class_label label

In [2]:
for col in df_feat_ictalVspreictal_1min.columns:
    print(col)

file_name
window_id
start_sample
end_sample
fs
n_channels
seizure_onsets
window_start_time
window_end_time
class_label
label_name
theta_power_EEG_SQ_P_SQ_C
theta_power_EEG_SQ_D_SQ_C
delta_power_EEG_SQ_P_SQ_C
rms_EEG_SQ_D_SQ_C
var_EEG_SQ_D_SQ_C
std_EEG_SQ_D_SQ_C


## 1. separate metadata from features

In [3]:
# Columns that are NOT EEG features
metadata_cols = [
    "file_name",
    "window_id",
    "start_sample",
    "end_sample",
    "fs",
    "n_channels",
    "window_sec",
    "seizure_onsets",
    "window_start_time",
    "window_end_time",
    "class_label",
    "label_name",
    "excluded_reason"
]

# Feature columns = everything except metadata
feature_cols = [col for col in df_feat_ictalVspreictal_1min.columns if col not in metadata_cols]

print("Number of feature columns:", len(feature_cols))
print(feature_cols)

Number of feature columns: 6
['theta_power_EEG_SQ_P_SQ_C', 'theta_power_EEG_SQ_D_SQ_C', 'delta_power_EEG_SQ_P_SQ_C', 'rms_EEG_SQ_D_SQ_C', 'var_EEG_SQ_D_SQ_C', 'std_EEG_SQ_D_SQ_C']


## 2. Build matrix 

In [4]:
X = df_feat_ictalVspreictal_1min[feature_cols].copy()
y = df_feat_ictalVspreictal_1min["class_label"].copy()

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (322, 6)
y shape: (322,)


## 3. check for NaNs or inf

In [5]:
import numpy as np

print("NaNs per column:")
print(X.isna().sum())

print("Any infinite values?")
print(np.isinf(X).sum().sum())

NaNs per column:
theta_power_EEG_SQ_P_SQ_C    0
theta_power_EEG_SQ_D_SQ_C    0
delta_power_EEG_SQ_P_SQ_C    0
rms_EEG_SQ_D_SQ_C            0
var_EEG_SQ_D_SQ_C            0
std_EEG_SQ_D_SQ_C            0
dtype: int64
Any infinite values?
0


## 4. Scale features before PCA
PCA needs features to be in comparable scale

In [6]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)

print("X_scaled shape:", X_scaled.shape)

X_scaled shape: (322, 6)


## 5. Apply PCA. start with only 5 components first

In [7]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# PCA with all possible components
pca_full = PCA()
X_pca_full = pca_full.fit_transform(X_scaled)

# Explained variance
explained_var = pca_full.explained_variance_ratio_
cumulative_var = np.cumsum(explained_var)

# Print explained variance per PC
for i, (var, cum_var) in enumerate(zip(explained_var, cumulative_var), start=1):
    print(f"PC{i}: {var*100:.2f}% | cumulative: {cum_var*100:.2f}%")

PC1: 94.22% | cumulative: 94.22%
PC2: 3.59% | cumulative: 97.81%
PC3: 1.43% | cumulative: 99.24%
PC4: 0.42% | cumulative: 99.66%
PC5: 0.34% | cumulative: 100.00%
PC6: 0.00% | cumulative: 100.00%


In [9]:
from sklearn.decomposition import PCA

pca = PCA(n_components=6)

X_pca = pca.fit_transform(X_scaled)

print("X_pca shape:", X_pca.shape)

X_pca shape: (322, 6)


In [11]:
explained_variance = pca.explained_variance_ratio_

for i, var in enumerate(explained_variance, start=1):
    print(f"PC{i}: {var:.4f} ({var*100:.2f}%)")

print("Total explained variance:", explained_variance.sum())

PC1: 0.9422 (94.22%)
PC2: 0.0359 (3.59%)
PC3: 0.0143 (1.43%)
PC4: 0.0042 (0.42%)
PC5: 0.0034 (0.34%)
PC6: 0.0000 (0.00%)
Total explained variance: 1.0


## 6. Final dataframe: windows x PCA

In [12]:
import pandas as pd

pca_cols = [f"PC{i+1}" for i in range(X_pca.shape[1])]

df_pca = pd.DataFrame(
    X_pca,
    columns=pca_cols,
    index=df_feat_ictalVspreictal_1min.index
)

df_pca.head()

,PC1,PC2,PC3,PC4,PC5,PC6
0,-0.734621,-0.122934,-0.070754,-0.020583,-0.026913,0.000051
1,-0.312641,0.198034,0.110246,0.010508,0.001074,0.000107
2,-0.955131,-0.116742,-0.253138,-0.035408,-0.059253,0.000045
3,-1.000685,-0.147267,-0.238189,-0.044670,-0.072222,0.000025
4,-0.672869,-0.060761,-0.053653,-0.007662,-0.026872,0.000027


## 7. ADD metadata again

In [15]:
# Keep only metadata columns that actually exist in the dataframe
metadata_cols_available = [
    col for col in metadata_cols 
    if col in df_feat_ictalVspreictal_1min.columns
]

# Optional: show which columns were ignored
metadata_cols_missing = [
    col for col in metadata_cols 
    if col not in df_feat_ictalVspreictal_1min.columns
]

print("Using metadata columns:", metadata_cols_available)
print("Ignored missing columns:", metadata_cols_missing)

# Concatenate available metadata columns with PCA dataframe
df_windows_pca = pd.concat(
    [
        df_feat_ictalVspreictal_1min[metadata_cols_available].reset_index(drop=True),
        df_pca.reset_index(drop=True)
    ],
    axis=1
)

df_windows_pca.head()

Using metadata columns: ['file_name', 'window_id', 'start_sample', 'end_sample', 'fs', 'n_channels', 'seizure_onsets', 'window_start_time', 'window_end_time', 'class_label', 'label_name']
Ignored missing columns: ['window_sec', 'excluded_reason']


,file_name,window_id,start_sample,end_sample,fs,n_channels,seizure_onsets,window_start_time,window_end_time,class_label,label_name,PC1,PC2,PC3,PC4,PC5,PC6
0,JYXFE_22_preproc_full.npz,983,2034810,2036880,207.031055,2,[2021-05-04 20:33:31.000000],2021-05-04 20:27:26.525499679,2021-05-04 20:27:36.523999679,1,preictal,-0.734621,-0.122934,-0.070754,-0.020583,-0.026913,0.000051
1,JYXFE_22_preproc_full.npz,984,2036880,2038950,207.031055,2,[2021-05-04 20:33:31.000000],2021-05-04 20:27:36.523999679,2021-05-04 20:27:46.522499678,1,preictal,-0.312641,0.198034,0.110246,0.010508,0.001074,0.000107
2,JYXFE_22_preproc_full.npz,985,2038950,2041020,207.031055,2,[2021-05-04 20:33:31.000000],2021-05-04 20:27:46.522499678,2021-05-04 20:27:56.520999678,1,preictal,-0.955131,-0.116742,-0.253138,-0.035408,-0.059253,0.000045
3,JYXFE_22_preproc_full.npz,986,2041020,2043090,207.031055,2,[2021-05-04 20:33:31.000000],2021-05-04 20:27:56.520999678,2021-05-04 20:28:06.519499678,1,preictal,-1.000685,-0.147267,-0.238189,-0.044670,-0.072222,0.000025
4,JYXFE_22_preproc_full.npz,987,2043090,2045160,207.031055,2,[2021-05-04 20:33:31.000000],2021-05-04 20:28:06.519499678,2021-05-04 20:28:16.517999678,1,preictal,-0.672869,-0.060761,-0.053653,-0.007662,-0.026872,0.000027


In [16]:
print(df_windows_pca.shape)
print(df_windows_pca.columns)

(322, 17)
Index(['file_name', 'window_id', 'start_sample', 'end_sample', 'fs',
       'n_channels', 'seizure_onsets', 'window_start_time', 'window_end_time',
       'class_label', 'label_name', 'PC1', 'PC2', 'PC3', 'PC4', 'PC5', 'PC6'],
      dtype='object')


In [17]:
# Sanity check:Same number of rows as original df
print(df_feat_ictalVspreictal_1min.shape[0] == df_windows_pca.shape[0])

True


## 8. export as pkl

In [18]:
from pathlib import Path

output_dir = Path("/home/tperezsanchez/FoundationModel_EEG_Dissertation/Main_project/results/JYXFE/Feature_ext/Part2_features/JYXFE_IN-normalized_npz_FP-fullnpz_W10s_PRE6to5min_ICT0to1min_GAPasINT_FINAL-PREvsSEIZ_20260511_v01_FEAT-TIME-FREQ_20260511_v01/")
output_dir.mkdir(parents=True, exist_ok=True)

output_path = output_dir / "df_windows_FEATCLEANJYXFEpca.pkl"

df_windows_pca.to_pickle(output_path)

print(f"Saved PCA dataframe to: {output_path}")
print("Shape:", df_windows_pca.shape)

Saved PCA dataframe to: /home/tperezsanchez/FoundationModel_EEG_Dissertation/Main_project/results/JYXFE/Feature_ext/Part2_features/JYXFE_IN-normalized_npz_FP-fullnpz_W10s_PRE6to5min_ICT0to1min_GAPasINT_FINAL-PREvsSEIZ_20260511_v01_FEAT-TIME-FREQ_20260511_v01/df_windows_FEATCLEANJYXFEpca.pkl
Shape: (322, 17)
